### Routing

#### Module 02: Routing Deep Dive

Master Axum's routing system with the new 0.8 syntax.

#### 🎯 What You'll Learn

- **New path syntax**: `/{id}` instead of `/:id`
- Wildcard routes with `/{*rest}`
- Query parameters
- Router nesting and merging
- HTTP method routing

#### 🚀 Running

```bash
cargo run
```

#### 📝 Endpoints

#### HTTP Methods Demo
| Method | Path | Description |
|--------|------|-------------|
| GET | `/resource` | Read all resources |
| POST | `/resource` | Create new resource |
| GET | `/resource/{id}` | Read single resource |
| PUT | `/resource/{id}` | Full update |
| PATCH | `/resource/{id}` | Partial update |
| DELETE | `/resource/{id}` | Remove resource |

#### Path Parameters & Nesting
| Method | Path | Description |
|--------|------|-------------|
| GET | `/users/{id}/posts/{post_id}` | Multiple path params |
| GET | `/files/{*path}` | Wildcard route |
| GET | `/items?page=1&limit=10` | Query params |
| GET/POST | `/api/v1/users` | Nested routes |
| GET/PUT/PATCH/DELETE | `/api/v1/users/{id}` | Full CRUD |

#### 💡 Key Changes in Axum 0.8

#### Path Parameters (NEW SYNTAX!)

```rust
// OLD (pre-0.8)
.route("/users/:id", get(handler))

// NEW (0.8+)
.route("/users/{id}", get(handler))
```

#### Wildcards (NEW SYNTAX!)

```rust
// OLD: "/*rest"
// NEW:
.route("/files/{*path}", get(files))
```

#### 🧪 Try It

```bash
# Path parameters
curl http://localhost:3000/users/123/posts/456

# Query parameters
curl "http://localhost:3000/items?page=2&limit=20"

# Nested routes
curl http://localhost:3000/api/v1/users
```

#### ▶️ Next Module

Continue to [Module 03: Extractors](../module-03-extractors)

### code

In [ ]:
//! # Module 02: Routing Deep Dive
//!
//! This module covers everything about routing in Axum 0.8:
//! - Path parameters with NEW `/{param}` syntax
//! - Wildcard routes with `/{*rest}`
//! - Query parameters
//! - Router nesting and merging
//! - Method routing (GET, POST, PUT, DELETE, etc.)

use axum::{
    extract::{Path, Query},
    routing::{delete, get, patch, post, put},
    Router,
};
use serde::Deserialize;

// ============================================================================
// LESSON 1: Path Parameters - NEW SYNTAX IN AXUM 0.8!
// ============================================================================

/// IMPORTANT: Axum 0.8 introduced new path syntax!
///
/// OLD (pre-0.8): `/:id` and `/*rest`
/// NEW (0.8+):    `/{id}` and `/{*rest}`
///
/// This change allows routes like `/api/:colon` that start with `:` or `*`

/// Single path parameter
async fn get_user(Path(id): Path<u64>) -> String {
    format!("Getting user with ID: {}", id)
}
/// Path is extractor -> exract data data from request path
/// id -> is the exetracted value we need 
/// <u64>  -> to define type of exetracted value 

/// Multiple path parameters - use a tuple
async fn get_user_post(Path((user_id, post_id)): Path<(u64, u64)>) -> String {
    format!("User {} - Post {}", user_id, post_id)
}
/// here we have more than value in path we want to extract it to use it 
/// Path((user_id, post_id)): Path<(u64, u64)>

/// for complex path use struct to 

/// You can also deserialize into a struct
#[derive(Deserialize)]
struct PostPath {
    user_id: u64,
    post_id: u64,
    comment_id: u64,
}

async fn get_comment(Path(params): Path<PostPath>) -> String {
    format!(
        "User {} - Post {} - Comment {}",
        params.user_id, params.post_id, params.comment_id
    )
}

// ============================================================================
// LESSON 2: Wildcard Routes - NEW SYNTAX!
// ============================================================================

/// Wildcard captures the rest of the path
/// OLD: `/*rest`
/// NEW: `/{*rest}`
async fn files(Path(path): Path<String>) -> String {
    format!("Accessing file: {}", path)
}

/// {*rest}
/// astrex (*) here mean any thing follow like .md .docx .png


// ============================================================================
// LESSON 3: Query Parameters
// ============================================================================

#[derive(Deserialize)]
struct Pagination {
    page: Option<u32>,
    limit: Option<u32>,
}
async fn list_items(Query(pagination): Query<Pagination>) -> String {
    let page = pagination.page.unwrap_or(1);
    let limit = pagination.limit.unwrap_or(10);
    format!("Listing items - Page: {}, Limit: {}", page, limit)
}
/// we use query extractor 
/// to extract limit - page 
/// 
/// 



/// Query parameters with validation
#[derive(Deserialize)]
struct SearchParams {
    q: String,
    category: Option<String>,
    sort: Option<String>,
}

async fn search(Query(params): Query<SearchParams>) -> String {
    format!(
        "Searching for '{}' in category '{}', sorted by '{}'",
        params.q,
        params.category.unwrap_or_else(|| "all".to_string()),
        params.sort.unwrap_or_else(|| "relevance".to_string())
    )
}

// ============================================================================
// LESSON 4: HTTP Methods
// ============================================================================

async fn create_user() -> &'static str {
    "Creating new user (POST)"
}

async fn update_user(Path(id): Path<u64>) -> String {
    format!("Updating user {} (PUT - full update)", id)
}

async fn patch_user(Path(id): Path<u64>) -> String {
    format!("Patching user {} (PATCH - partial update)", id)
}

async fn delete_user(Path(id): Path<u64>) -> String {
    format!("Deleting user {} (DELETE)", id)
}

// ============================================================================
// LESSON 5: Router Nesting
// ============================================================================

/// Create a sub-router for user-related routes
fn user_routes() -> Router {
    Router::new()
        .route("/", get(list_users).post(create_user))
        .route(
            "/{id}",
            get(get_user)
                .put(update_user)
                .patch(patch_user)
                .delete(delete_user),
        )
}

async fn list_users() -> &'static str {
    "Listing all users"
}

/// Create a sub-router for post-related routes
fn post_routes() -> Router {
    Router::new()
        .route("/", get(list_posts))
        .route("/{id}", get(get_post))
}

async fn list_posts() -> &'static str {
    "Listing all posts"
}

async fn get_post(Path(id): Path<u64>) -> String {
    format!("Getting post {}", id)
}

// ============================================================================
// LESSON 6: Router Merging
// ============================================================================

/// Create an API v1 router by merging multiple routers
fn api_v1_routes() -> Router {
    Router::new()
        .nest("/users", user_routes())
        .nest("/posts", post_routes())
}

/// You can also have multiple API versions
fn api_v2_routes() -> Router {
    Router::new()
        .route("/users", get(|| async { "API v2 - Users endpoint" }))
        .route("/posts", get(|| async { "API v2 - Posts endpoint" }))
}

// ============================================================================
// LESSON 7: Fallback Routes
// ============================================================================

async fn not_found() -> (axum::http::StatusCode, &'static str) {
    (axum::http::StatusCode::NOT_FOUND, "404 - Route not found")
}

// ============================================================================
// MAIN: Putting It All Together
// ============================================================================

#[tokio::main]
async fn main() {
    let app = Router::new()
        // Basic routes
        .route("/", get(|| async { "Welcome to the Routing Module!" }))
        // ===== HTTP METHODS DEMO =====
        // Each method demonstrated with a standalone route
        .route("/resource", get(|| async { "GET - Read resource" }))
        .route("/resource", post(|| async { "POST - Create resource" }))
        .route(
            "/resource/{id}",
            get(|Path(id): Path<u64>| async move { format!("GET - Read resource {}", id) }),
        )
        .route(
            "/resource/{id}",
            put(|Path(id): Path<u64>| async move { format!("PUT - Full update resource {}", id) }),
        )
        .route(
            "/resource/{id}",
            patch(|Path(id): Path<u64>| async move {
                format!("PATCH - Partial update resource {}", id)
            }),
        )
        .route(
            "/resource/{id}",
            delete(|Path(id): Path<u64>| async move { format!("DELETE - Remove resource {}", id) }),
        )
        // Path parameters (new syntax!)
        .route("/users/{id}/posts/{post_id}", get(get_user_post))
        .route(
            "/users/{user_id}/posts/{post_id}/comments/{comment_id}",
            get(get_comment),
        )
        // Wildcard route (must come after specific routes)
        .route("/files/{*path}", get(files))
        // Query parameters
        .route("/items", get(list_items))
        .route("/search", get(search))
        // Nested routers - creates /api/v1/users, /api/v1/posts, etc.
        .nest("/api/v1", api_v1_routes())
        .nest("/api/v2", api_v2_routes())
        // Fallback for unmatched routes
        .fallback(not_found);

    let listener = tokio::net::TcpListener::bind("0.0.0.0:3000")
        .await
        .expect("Failed to bind to port 3000");

    println!("🚀 Module 02: Routing Deep Dive");
    println!("   Server running on http://localhost:3000");
    println!();
    println!("📝 HTTP Methods Demo:");
    println!("   GET    /resource      - Read all");
    println!("   POST   /resource      - Create new");
    println!("   GET    /resource/123  - Read one");
    println!("   PUT    /resource/123  - Full update");
    println!("   PATCH  /resource/123  - Partial update");
    println!("   DELETE /resource/123  - Remove");
    println!();
    println!("📝 Path Parameters (NEW SYNTAX!):");
    println!("   GET /users/123/posts/456");
    println!("   GET /files/docs/readme.md");
    println!();
    println!("📝 Nested Routes:");
    println!("   GET  /api/v1/users");
    println!("   POST /api/v1/users");
    println!("   PUT  /api/v1/users/123");

    axum::serve(listener, app).await.expect("Server failed");
}

### Basic syntax

A closure is an anonymous, inline function that can **capture variables from the scope it's defined in** — something a regular `fn` can't do.



```rust
let add = |a, b| a + b;
println!("{}", add(2, 3)); // 5
```

- `|a, b|` — parameter list, using pipes `|...|` instead of parens.
- `a + b` — the body. If it's a single expression, no `{}` or `return` needed.
- Types are usually **inferred**, but you can annotate them if needed:

```rust
let add = |a: i32, b: i32| -> i32 { a + b };
```

Compare to a regular function:

```rust
fn add(a: i32, b: i32) -> i32 { a + b }
let add = |a: i32, b: i32| -> i32 { a + b };
```

Same job, different form. The big difference isn't syntax — it's **capturing**.

#### Capturing the environment

This is what makes closures special:

```rust
let factor = 10;
let multiply = |x| x * factor; // captures `factor` from outside
println!("{}", multiply(5)); // 50
```

A plain `fn` could never do this — `factor` is a local variable, and a `fn` has no access to anything outside its own parameter list. A closure can "see" and use variables from its surrounding scope.

#### Three ways a closure can capture

Rust picks automatically based on how the closure body uses the variable:

```rust
let s = String::from("hello");

// 1. Borrow immutably (Fn) — closure just reads s
let read_it = || println!("{}", s);
read_it();
println!("{}", s); // s still usable here

// 2. Borrow mutably (FnMut) — closure modifies something
let mut count = 0;
let mut increment = || { count += 1; };
increment();

// 3. Take ownership (FnOnce, or forced with `move`)
let owned = move || println!("{}", s);
// s is moved into the closure now — can't use s after this line
```

This maps directly onto the borrowing rules you already know from ownership: a closure that only reads captures by reference, one that mutates captures by mutable reference, and `move` forces it to take ownership outright — the same &T / &mut T / T distinction, just applied to closure captures instead of function arguments.

#### Why `move` matters (tying back to your axum code)

```rust
get(|Path(id): Path<u64>| async move { format!("GET - Read resource {}", id) })
```

`id` is a value the closure receives as its own parameter. The `async move` block needs to *own* `id` because the `Future` it produces might be polled later, after the closure call has already returned — a borrow wouldn't live long enough. `move` copies/transfers ownership of `id` into the `Future`, so it's self-contained and safe to run whenever the runtime gets to it.

#### Closures as values

The other key property: closures are values you can store, pass around, and return — which is exactly why axum accepts them as handlers:

```rust
fn make_adder(x: i32) -> impl Fn(i32) -> i32 {
    move |y| x + y
}

let add5 = make_adder(5);
println!("{}", add5(10)); // 15
```

`add5` is a closure that "remembers" `x = 5` — this is a closure acting like a mini-object bundling data (`x`) with behavior (`+ y`).

#### Quick reference table

| Trait | Capture mode | Can call... |
|---|---|---|
| `Fn` | borrows (`&T`) | multiple times |
| `FnMut` | mutably borrows (`&mut T`) | multiple times, needs `mut` binding |
| `FnOnce` | takes ownership (`T`) | only once |

`move` doesn't force `FnOnce` — it just forces *ownership* of captured variables; the closure can still implement `Fn`/`FnMut` if it doesn't consume them internally. That's exactly why `async move { format!(...) }` works fine even though axum may call the handler closure repeatedly across requests — each call produces a fresh Future that owns its own copy of `id`.

Here's a walkthrough of the whole module, lesson by lesson.

---

#### Lesson 1: Path Parameters (new `{param}` syntax)

Axum 0.8 changed the path-parameter syntax from `:id` to `{id}`. This was done specifically so a literal `:` or `*` could appear in a route segment without ambiguity (e.g. `/api/:colon` as a real path).

```rust
async fn get_user(Path(id): Path<u64>) -> String {
    format!("Getting user with ID: {}", id)
}
```

`Path<T>` is an **extractor** — a type that knows how to pull data out of the incoming request. Here it grabs the captured segment from the URL and tries to parse it as a `u64`. If the URL has `/users/abc` (not a number), axum automatically returns a `400 Bad Request` before your handler even runs — you don't have to check that yourself.

For **multiple** params, you extract into a tuple:

```rust
async fn get_user_post(Path((user_id, post_id)): Path<(u64, u64)>) -> String
```

And for **many/complex** params, you extract into a struct that derives `Deserialize` — axum matches struct field names to path segment names:

```rust
#[derive(Deserialize)]
struct PostPath { user_id: u64, post_id: u64, comment_id: u64 }

async fn get_comment(Path(params): Path<PostPath>) -> String
```

This is nicer than a big tuple because the fields are named, not positional.

#### Lesson 2: Wildcards (`{*rest}`)

```rust
async fn files(Path(path): Path<String>) -> String {
    format!("Accessing file: {}", path)
}
```

`{*path}` matches **everything** after that point in the URL, including additional `/` — so `/files/docs/readme.md` gives `path = "docs/readme.md"`. Note it's captured as a single `String`, not split up — the `*` swallows the whole remaining tail. Wildcard routes must be registered carefully since they're greedy (the file's comment about ordering matters — more specific routes should generally be matched before a catch-all).

#### Lesson 3: Query Parameters

```rust
#[derive(Deserialize)]
struct Pagination { page: Option<u32>, limit: Option<u32> }

async fn list_items(Query(pagination): Query<Pagination>) -> String {
    let page = pagination.page.unwrap_or(1);
    ...
}
```

`Query<T>` parses the `?key=value&key2=value2` part of the URL into your struct. Making fields `Option<T>` means they're optional — if the client doesn't send `?page=2`, you just get `None` and fall back to a default with `unwrap_or`. If a field were *not* `Option`, a missing query param would cause a `400` instead.

`SearchParams` does the same thing but with a required `String` field (`q`) and optional ones — the required field would 400 if the client forgot `?q=...`.

#### Lesson 4: HTTP Methods

Nothing new mechanically here — just handlers for each verb (`create_user`, `update_user`, `patch_user`, `delete_user`). The interesting part is how they get *wired to methods*, shown next.

#### Lesson 5: Router Nesting

```rust
fn user_routes() -> Router {
    Router::new()
        .route("/", get(list_users).post(create_user))
        .route("/{id}", get(get_user).put(update_user).patch(patch_user).delete(delete_user))
}
```

Notice `.route(path, method_router)` — you chain `.post()`, `.put()`, etc. onto the same path builder to attach multiple HTTP methods to one route. So `GET /` and `POST /` on the same path are two branches of the *same* `.route()` call, not two separate ones. `post_routes()` follows the identical pattern for posts.

`user_routes()` and `post_routes()` return standalone `Router`s — they're self-contained and don't know they'll later be mounted under a prefix.

#### Lesson 6: Merging via `.nest()`

```rust
fn api_v1_routes() -> Router {
    Router::new()
        .nest("/users", user_routes())
        .nest("/posts", post_routes())
}
```

`.nest("/users", user_routes())` takes the router built in Lesson 5 (which only knows about `/` and `/{id}`) and mounts it under `/users`, so its routes effectively become `/users/` and `/users/{id}`. This is how you compose small routers into bigger ones without hardcoding full paths everywhere.

`api_v2_routes()` shows the same idea done inline with closures instead of separate functions — just a different style, same mechanism.

#### Lesson 7: Fallback

```rust
async fn not_found() -> (axum::http::StatusCode, &'static str) {
    (axum::http::StatusCode::NOT_FOUND, "404 - Route not found")
}
```

Tuple return types like `(StatusCode, &'static str)` implement `IntoResponse` — the first element sets the status code, the second becomes the body. `.fallback(not_found)` registers this as the handler for any request that doesn't match any defined route.

#### `main()`: assembling everything

```rust
let app = Router::new()
    .route("/", get(|| async { "..." }))
    .route("/resource", get(...))
    .route("/resource", post(...))
    ...
    .nest("/api/v1", api_v1_routes())
    .nest("/api/v2", api_v2_routes())
    .fallback(not_found);
```

A few things worth noticing:

- **Inline closures as handlers**: `get(|| async { "Welcome..." })` — you don't always need a named `async fn`; a closure returning an `async` block works the same way as long as it matches axum's handler trait bounds.
- **Repeated `.route("/resource", ...)` calls with different methods** — unlike Lesson 5, here each verb is registered in a *separate* `.route()` call rather than chained. Both styles work; chaining (`get(x).post(y)`) is just more concise when it's the same handler pattern.
- **Route ordering with wildcards**: `/files/{*path}` is declared after the more specific routes — generally a good habit, since a wildcard could otherwise "steal" matches meant for a more specific path (axum's router actually handles precedence internally by specificity, but keeping wildcards visually last avoids confusion when reading the code).
- **`tokio::net::TcpListener::bind(...).await.expect(...)`**: opens the actual TCP socket. `axum::serve(listener, app).await` then starts accepting connections and dispatching them into your router.

The overall structure this module is teaching: **extractors** (`Path`, `Query`) pull typed data out of a request, **method routers** (`get`, `post`, etc.) map an HTTP verb to a handler function, and `.nest()`/`.merge()` let you build a big API out of small, independently-testable sub-routers.


---


Both of these are **inline closures** used as handlers instead of separate `async fn`s. Let's break down each piece.

#### Simple version: `|| async { "..." }`

```rust
.route("/users", get(|| async { "API v2 - Users endpoint" }))
```

This is equivalent to writing:

```rust
async fn users_handler() -> &'static str {
    "API v2 - Users endpoint"
}
.route("/users", get(users_handler))
```

Breaking down `|| async { "..." }`:

- `||` — a closure that takes **no arguments** (empty parameter list, same as `fn foo()`).
- `async { ... }` — an **async block**. On its own, `async { ... }` doesn't run anything immediately — it produces a `Future` that yields the block's value when awaited. Combining `||` with `async { }` gives you a closure that, when called, *returns a Future* — which is exactly the shape axum's handler trait expects (handler functions must be async, i.e. return something `Future`-like).

So `|| async { "text" }` is really "a function with no args that returns a Future<Output = &'static str>" — functionally identical to `async fn handler() -> &'static str { "text" }`, just written as an anonymous closure instead of a named function.

Why use a closure here instead of a named fn? It's purely a style choice for small, throwaway handlers you won't reuse — you save yourself naming a function like `api_v2_users` for one line of logic. If the logic grows, you'd normally switch back to a named `async fn`.

#### With an extractor: `|Path(id): Path<u64>| async move { ... }`

```rust
get(|Path(id): Path<u64>| async move { format!("GET - Read resource {}", id) })
```

Same idea, but now the closure takes a **parameter**:

- `|Path(id): Path<u64>|` — the closure takes one argument. Just like in the named-function version (`Path(id): Path<u64>`), this is a **destructuring pattern**: the argument's type is `Path<u64>`, and `Path(id)` pattern-matches into it to pull out the inner `u64` and bind it to `id`. Axum sees the closure's parameter type (`Path<u64>`) and knows to run the `Path` extractor on the incoming request, then pass the result in.

- `async move { ... }` — this is the closure's *body*. It's an async block again, but this time with `move`.

#### Why `move` here but not in the first example?

`move` forces the closure to take ownership of any variables it uses from its environment, instead of borrowing them.

In the simple example (`|| async { "text" }`), the async block doesn't reference anything outside itself — no `move` needed.

In this one, the async block uses `id`, which was captured by the *closure* from its argument. Here's the subtlety: the closure itself runs synchronously and immediately returns a `Future` (the async block) — but that `Future` might not actually be polled/executed until *later* (once axum's runtime gets around to it). If the async block just *borrowed* `id`, that borrow would need to outlive the closure call itself, which isn't guaranteed — `id` is a local variable that would otherwise go out of scope right after the closure returns. `move` says: "take ownership of `id` into the Future itself," so the Future carries its own copy and stays valid regardless of when it's polled.

**Rule of thumb:** any time an async block/closure needs to hang onto a variable across an `.await` point or return a `Future` that outlives the current scope, use `async move`. You'll see this constantly with async closures in Rust — it's almost always required whenever the block captures anything.

#### Side-by-side with the named-function equivalent

```rust
// closure version:
get(|Path(id): Path<u64>| async move { format!("GET - Read resource {}", id) })

// named-fn version (identical behavior):
async fn get_resource(Path(id): Path<u64>) -> String {
    format!("GET - Read resource {}", id)
}
get(get_resource)
```

They compile to the same thing — axum just needs something matching its `Handler` trait, and both an `async fn` and a closure returning `async move { }` qualify.
